In [1]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np
from scipy.stats import binomtest
from engine import *
from research_stats import *

df = load_clean()
cfg = Config(threshold=-0.02, hold=5)
ev_all, base_all = get_events(df, cfg), get_baseline(df, cfg)

def report(name, ev, base):
    r = diff_test(ev["fwd_ret"], base["fwd_ret"], block=20, n_boot=2000)
    print(f"{name:20s} n={len(ev):3d} ev={ev['fwd_ret'].mean()*100:6.3f}% "
          f"base={base['fwd_ret'].mean()*100:6.3f}% diff={r['diff_%']:6.3f}% "
          f"CI=[{r['ci_lo_%']:.2f}, {r['ci_hi_%']:.2f}]")

report("Full sample", ev_all, base_all)
for label, yrs in [("Excl. 2008", [2008]), ("Excl. 2020", [2020]), ("Excl. 2008 & 2020", [2008, 2020])]:
    report(label, ev_all[~ev_all.index.year.isin(yrs)], base_all[~base_all.index.year.isin(yrs)])

k, n = (ev_all["fwd_ret"] > 0).sum(), len(ev_all)
p0 = (base_all["fwd_ret"] > 0).mean()
print("Event win rate:", round(k / n, 3), "| baseline:", round(p0, 3),
      "| sign-test p:", round(binomtest(k, n, p0, alternative="greater").pvalue, 3))

Full sample          n=131 ev= 0.017% base= 0.117% diff=-0.100% CI=[-0.88, 0.71]
Excl. 2008           n=104 ev= 0.477% base= 0.188% diff= 0.290% CI=[-0.51, 1.10]
Excl. 2020           n=117 ev= 0.170% base= 0.104% diff= 0.067% CI=[-0.74, 0.91]
Excl. 2008 & 2020    n= 90 ev= 0.748% base= 0.178% diff= 0.571% CI=[-0.16, 1.31]
Event win rate: 0.496 | baseline: 0.54 | sign-test p: 0.861


Diagnostics only, run after seeing OOS. They cannot change the conclusion. They only show whether the result depends on the crash years

Excluding 2008 and 2020 flips the sign of the difference (+0.57%) but the CI still includes 0.
This subgroup was chosen after seeing results (post-hoc), so it is exploratory only.
Evidence that would make me reject the hypothesis: CI for the difference including 0 in
both dev and OOS (observed), and a negative net mean after costs (observed).